# Qwen2.5 Coder C++ Review QLoRA Training on Kaggle

Run this notebook from top to bottom. It copies your read-only Kaggle project files into `/kaggle/working`, configures the dataset path, installs dependencies with `uv`, and starts training. Re-running the training cell resumes automatically from the latest checkpoint.

In [ ]:
!nvidia-smi
!python --version
!python -m pip install --upgrade uv

## Copy Project to Writable Storage

Kaggle mounts `/kaggle/input` as read-only. Training needs to write configs, checkpoints, adapters, `.pth` files, and ONNX exports, so the project is copied to `/kaggle/working/project-files`.

In [ ]:
from pathlib import Path
import shutil

PROJECT_INPUT = Path('/kaggle/input/datasets/saffiullah892/project-files')
WORK_REPO = Path('/kaggle/working/project-files')
DATASET_FILE = Path('/kaggle/input/datasets/saffiullah892/clean-datset01/merged_cleaned.jsonl')

assert PROJECT_INPUT.exists(), f'Project folder not found: {PROJECT_INPUT}'
assert DATASET_FILE.exists(), f'Training dataset not found: {DATASET_FILE}'

shutil.copytree(PROJECT_INPUT, WORK_REPO, dirs_exist_ok=True)
%cd /kaggle/working/project-files

assert Path('pyproject.toml').exists(), 'pyproject.toml missing after copy'
print('Project copied to:', WORK_REPO)
print('Training dataset:', DATASET_FILE)

## Install Dependencies with uv

In [ ]:
!mkdir -p /kaggle/temp/uv-cache /kaggle/temp/project-venv /kaggle/temp/hf-cache /kaggle/temp/hf-datasets
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv UV_CACHE_DIR=/kaggle/temp/uv-cache UV_LINK_MODE=copy uv sync --extra gpu --extra export --extra dev
!du -sh /kaggle/temp/project-venv /kaggle/temp/uv-cache /kaggle/working/project-files || true

## Configure Training

This cell writes your Kaggle dataset path into `configs/train_qlora.yaml` and sends all outputs to `/kaggle/working/outputs`, which is writable and downloadable.

In [ ]:
import yaml
from pathlib import Path

DATASET_FILE = '/kaggle/input/datasets/saffiullah892/clean-datset01/merged_cleaned.jsonl'
OUTPUT_DIR = '/kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora'

config_path = Path('configs/train_qlora.yaml')
config = yaml.safe_load(config_path.read_text())

config['data']['data_files'] = [DATASET_FILE]
config['data']['cache_dir'] = '/kaggle/temp/hf-datasets'
config['data']['identifier_augmentation'] = True
config['data']['identifier_augmentation_copies'] = 1
config['training']['output_dir'] = OUTPUT_DIR
config['training']['resume_from_checkpoint'] = None
config['training']['logging_steps'] = 1
config['training']['packing'] = False
config['training']['gradient_checkpointing_use_reentrant'] = False
config['training']['ddp_find_unused_parameters'] = False
config['training']['save_total_limit'] = 2
config['training']['save_steps'] = 250
config['training']['eval_steps'] = 250

config_path.write_text(yaml.safe_dump(config, sort_keys=False))

print('Configured dataset:', config['data']['data_files'])
print('Configured output:', config['training']['output_dir'])

## Optional Sanity Check

This checks that the dataset file is readable and shows the first row keys.

In [ ]:
import json
from pathlib import Path

dataset_path = Path('/kaggle/input/datasets/saffiullah892/clean-datset01/merged_cleaned.jsonl')
with dataset_path.open() as handle:
    first = json.loads(handle.readline())

print('Dataset size GB:', round(dataset_path.stat().st_size / 1024**3, 3))
print('First row keys:', sorted(first.keys()))

## Start or Resume Training

This cell detects the number of GPUs and chooses single-GPU or multi-GPU Accelerate launch. If checkpoints already exist in `OUTPUT_DIR`, `train.py` resumes from the newest `checkpoint-*` automatically.

In [ ]:
%%bash
set -euo pipefail
export PYTHONUNBUFFERED=1
export HF_HOME=/kaggle/temp/hf-cache
export HF_DATASETS_CACHE=/kaggle/temp/hf-datasets
export TRANSFORMERS_CACHE=/kaggle/temp/hf-cache
export UV_CACHE_DIR=/kaggle/temp/uv-cache
export UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv
export ACCELERATE_LOG_LEVEL=info
export TRANSFORMERS_VERBOSITY=info
export NO_COLOR=1
LOG_FILE=/kaggle/working/train.log
echo "Training log: ${LOG_FILE}"
echo "Started at: $(date)" | tee -a "${LOG_FILE}"

NUM_GPUS=$(uv run python - <<'PY'
import torch
print(torch.cuda.device_count())
PY
)

echo "Detected GPUs: ${NUM_GPUS}" | tee -a "${LOG_FILE}"

if [ "${NUM_GPUS}" -gt 1 ]; then
  uv run accelerate launch \
    --config_file configs/accelerate_multi_gpu.yaml \
    --num_processes "${NUM_GPUS}" \
    train.py --config configs/train_qlora.yaml 2>&1 | tee -a "${LOG_FILE}"
else
  uv run accelerate launch \
    --config_file configs/accelerate_single_gpu.yaml \
    train.py --config configs/train_qlora.yaml 2>&1 | tee -a "${LOG_FILE}"
fi

## View Training Log

Run this after training finishes, or from the Kaggle console while training is running.

In [ ]:
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv tail -n 80 /kaggle/working/train.log || true

## Check Saved Training Outputs

Expected outputs include `best_adapter/`, `best_adapter.pth`, `last_adapter/`, `last_adapter.pth`, `final_adapter/`, and `final_adapter.pth`.

In [ ]:
!ls -lah /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora || true
!find /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora -maxdepth 2 -type f \( -name '*.pth' -o -name 'adapter_model.safetensors' -o -name 'trainer_state.json' \) -print || true

## Evaluate Best Adapter

In [ ]:
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv UV_CACHE_DIR=/kaggle/temp/uv-cache uv run python evaluate.py \
  --config configs/train_qlora.yaml \
  --adapter /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora/best_adapter

## Merge Best LoRA Adapter

In [ ]:
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv UV_CACHE_DIR=/kaggle/temp/uv-cache uv run python merge_lora.py \
  --base-model Qwen/Qwen2.5-Coder-1.5B-Instruct \
  --adapter /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-qlora/best_adapter \
  --output-dir /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-merged

## Export Merged Model to ONNX

In [ ]:
!UV_PROJECT_ENVIRONMENT=/kaggle/temp/project-venv UV_CACHE_DIR=/kaggle/temp/uv-cache uv run python export_onnx.py \
  --model /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review-merged \
  --output /kaggle/working/outputs/qwen2.5-coder-1.5b-cpp-review.onnx

## Final Files to Download

Download from `/kaggle/working/outputs` after training finishes.

In [ ]:
!find /kaggle/working/outputs -maxdepth 3 -type f \( -name '*.pth' -o -name '*.onnx' -o -name 'adapter_model.safetensors' -o -name 'model.safetensors' -o -name 'training_config.yaml' \) -print || true